# <p style="font-family:newtimeroman;font-size:150%;text-align:center;color:#06445e;">Author: Mohammed Hamid </p>


# <p style="font-family:newtimeroman;font-size:110%;text-align:center;color:#06445e;">Phase - 1.2 Pandas: Topic - 17: Reshaping Data</p>


### **Reshaping Data**

### **1. The intuition**

Imagine a table showing sales in three months:

| Product | Jan | Feb | Mar |
|---------|----:|----:|----:|
| Laptop  | 10  | 15  | 12  |
| Phone   | 20  | 18  | 25  |

This format is called **`wide format`**. Each month has its own column.


But suppose you want to analyze sales by month, group them, or plot a time trend. A long format is often easier:

| Product | Month | Sales |
|---------|-------|------:|
| Laptop  | Jan   | 10 |
| Laptop  | Feb   | 15 |
| Laptop  | Mar   | 12 |
| Phone   | Jan   | 20 |
| Phone   | Feb   | 18 |
| Phone   | Mar   | 25 |


Same information. Different shape.

### **2. The four tools in this topic**

| Tool | Main purpose |
|------|---------------|
| `melt()` | Wide → long |
| `pivot()` | Long → wide |
| `stack()` | Move column labels into the index |
| `unstack()` | Move index levels into columns |


### **3. First exercise — melt()**

In [15]:
import pandas as pd
sales_wide = pd.DataFrame({
    "Product": ["Laptop", "Phone"],
    "Jan": [10, 20],
    "Feb": [15, 18],
    "Mar": [12, 25]
})

display(sales_wide)

,Product,Jan,Feb,Mar
0,Laptop,10,15,12
1,Phone,20,18,25


Before writing code, predict:
1. Which column should remain an identifier?
2. Which columns should become values in a new Month column?
3. After reshaping, how many rows should the DataFrame contain?

In [16]:
sales_long = sales_wide.melt(
    id_vars=['Product'],
    value_vars=['Jan', 'Feb', 'Mar'],
    var_name='Month',
    value_name='Sales'
)
print(sales_long)
print(sales_long.shape)

  Product Month  Sales
0  Laptop   Jan     10
1   Phone   Jan     20
2  Laptop   Feb     15
3   Phone   Feb     18
4  Laptop   Mar     12
5   Phone   Mar     25
(6, 3)


1. Product
2. Jan, Feb, and Mar
3. 6

You can omit value_vars because Pandas automatically melts every column not listed in id_vars:

In [17]:
sales_long = sales_wide.melt(
    id_vars=["Product"],
    var_name="Month",
    value_name="Sales"
)

Both versions are correct. Explicitly writing value_vars can make your intention clearer.

### **Pivot()**

In [18]:
sales_wide_again = sales_long.pivot(
    index="Product",
    columns="Month",
    values="Sales"
)

print(sales_wide_again)

Month    Feb  Jan  Mar
Product               
Laptop    15   10   12
Phone     18   20   25


### **pivot() vs pivot_table()**

The key idea:
- pivot() needs one value for each row-and-column combination.
- pivot_table() can handle repeated combinations by summarizing them—for example, taking their average or adding them together.

In [19]:
sales_repeated = pd.DataFrame({
    "Product": ["Laptop", "Laptop", "Phone", "Phone"],
    "Month": ["Jan", "Jan", "Jan", "Feb"],
    "Sales": [10, 14, 20, 18]
})

print(sales_repeated)

  Product Month  Sales
0  Laptop   Jan     10
1  Laptop   Jan     14
2   Phone   Jan     20
3   Phone   Feb     18


Notice: Laptop + Jan appears twice—with sales of 10 and 14.

### **Your challenge**
Before running anything, predict:
1. What will happen if we use pivot() on this data? Will it work, or raise an error? Why?
2. If we use pivot_table() with aggfunc="mean", what will Laptop's January value be?
3. If we use pivot_table() with aggfunc="sum", what will Laptop's January value be?

### **Predictions**
1. It will raise error because there are some repeadted rows and pivot need unique combination of value and index 
2. 12
3. 24

In [20]:
sales_repeated.pivot(
    index="Product",
    columns="Month",
    values= "Sales"
)

ValueError: Index contains duplicate entries, cannot reshape

In [ ]:
(sales_repeated.pivot_table(
    index="Product",
    columns="Month",
    values= "Sales",
    aggfunc="mean"
))

Month,Feb,Jan
Product,,
Laptop,NaN,12.0
Phone,18.0,20.0


### **Remember this distinction**
| Method | Purpose |
|---|---|
| `melt()` | Wide → long |
| `pivot()` | Long → wide; requires unique combinations |
| `pivot_table()` | Long → wide; aggregates repeated combinations |

### **stack() and unstack()**

Before writing code, predict:
1. How many data values are currently in the table?
2. After stacking the month columns, how many rows will the result have?
3. What will the new index contain?

1. 6
2. 6
3. product + Month

In [ ]:
sales_stacked = sales_wide_again.stack()

print(sales_stacked)
print(sales_stacked.shape)

Product  Month
Laptop   Feb      15
         Jan      10
         Mar      12
Phone    Feb      18
         Jan      20
         Mar      25
dtype: int64
(6,)


In [ ]:
sales_unstacked = sales_stacked.unstack()

print(sales_unstacked)
print(sales_unstacked.shape)

Month    Feb  Jan  Mar
Product               
Laptop    15   10   12
Phone     18   20   25
(2, 3)


In [ ]:
sales_unstacked.equals(sales_wide_again)

True

### **You have now practiced:**
- melt() — wide to long
- pivot() — long to wide with unique combinations
- pivot_table() — long to wide with aggregation
- stack() — columns into index
- unstack() — index level into columns

### **Tidy Data Principles**

`idy data means organizing a dataset so its structure is clear and predictable.`



1. **Each variable gets its own column.**  
    For example, `Product`, `Month`, and `Sales` should be separate columns.

2. **Each observation gets its own row.**  
    For example, one row should represent Laptop sales for January.

3. **Each type of observational unit gets its own table.**  
    Keep customer details in a customer table and individual orders in an orders table rather than mixing unrelated units.

### Compare these structures

**Untidy — months are spread across columns**

| Product | Jan | Feb |
|---|---:|---:|
| Laptop | 10 | 15 |
| Phone | 20 | 18 |

**Tidy — each month is a value in one column**

| Product | Month | Sales |
|---|---|---:|
| Laptop | Jan | 10 |
| Laptop | Feb | 15 |
| Phone | Jan | 20 |
| Phone | Feb | 18 |

### **Your exercise**

Before writing code, predict:

1. What does one row represent in the wide table?
2. If we reshape the table into tidy long format, what should the three columns be called?
3. How many rows should the tidy result contain?

In [22]:
survey_wide = pd.DataFrame({
    "Student": ["A", "B"],
    "Math": [80, 90],
    "Science": [85, 95]
})

survey_wide

,Student,Math,Science
0,A,80,85
1,B,90,95


1. One student's record.
2. 3 columns. Student, Subject, and Marks
3. 4 rows.

In [29]:
print(survey_wide.melt(
    id_vars="Student",
    value_name="Marks",
    var_name="Subject",
    value_vars=["Math", "Science"]
))



  Student  Subject  Marks
0       A     Math     80
1       B     Math     90
2       A  Science     85
3       B  Science     95
